# MicCheck: Measuring and Removing Recording-Channel Shortcuts in Speech Representations

**Natural paired-channel probe:** the AMI Meeting Corpus provides closely corresponding speech segments recorded with an
Individual Headset Microphone (IHM) and a Single Distant Microphone (SDM). MicCheck asks how much an acoustic
representation moves when the recording condition changes, whether that movement harms useful speech information, and
whether conventional normalization or paired representation learning can reduce it.

### Research questions

1. How accurately can a classifier recover IHM versus SDM from lightweight acoustic features?
2. How much does useful speaker and lexical information degrade across recording conditions?
3. Which normalization gives the best tradeoff between low channel leakage and retained task information?
4. Can matched recordings train a compact channel-invariant representation?
5. Does an approximate IHM→SDM spectral transformation shift a microphone classifier?

This notebook is deliberately CPU-friendly. It streams a bounded subset, processes one recording at a time, caches only
features, and uses no pretrained speech foundation model.


## Hypotheses and scientific scope

- **H1:** Conventional acoustic features encode substantial recording-condition information.
- **H2:** Closed-set speech-information probes perform better within a recording condition than across conditions.
- **H3:** Channel normalization removes some condition information without eliminating useful information.
- **H4:** A pairwise IHM/SDM constraint yields more channel-invariant embeddings.

IHM and SDM differ in distance, reverberation, noise, placement, and hardware. Accordingly, this study measures
**recording-channel sensitivity and shortcut information**. It is not proof of a microphone-only causal effect.


## 1. Environment setup

Install the repository requirements before running the notebook. `datasets` is used in streaming mode. Audio decoding is
requested with `decode=False` and handled through `soundfile`, avoiding a hard dependency on `torchcodec` when possible.


In [ ]:
# Run once if this environment has not been prepared from requirements.txt:
# %pip install -r requirements.txt
print("Environment setup: use requirements.txt for reproducible installation.")


In [ ]:
import gc, hashlib, io, json, os, random, re, shutil, sys, warnings
from collections import Counter, defaultdict
from pathlib import Path

import datasets, fsspec, librosa, matplotlib, matplotlib.pyplot as plt
import numpy as np, pandas as pd, scipy, soundfile as sf, sklearn, torch
from datasets import Audio as HFAudio, load_dataset
from IPython.display import Audio, Markdown, display
from scipy import ndimage, signal
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score, roc_auc_score)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.svm import SVC
from tqdm.auto import tqdm

print("Python", sys.version.split()[0])
for package in (datasets, np, pd, scipy, librosa, sklearn, matplotlib, torch):
    print(f"{package.__name__:12s} {package.__version__}")


## 2. Configuration and reproducibility

`smoke` is for debugging, `standard` is the recommended run, and `extended` trades more time for a larger sample.
All caches and results use repository-relative paths. The cache signature includes the settings that affect pairing and features.


In [ ]:
SEED = 42
RUN_MODE = "standard"  # "smoke", "standard", or "extended"
MAX_PAIRS = {"smoke": 200, "standard": 2000, "extended": 5000}[RUN_MODE]
AMI_SPLIT = "validation"
MIN_DURATION, MAX_DURATION = 0.5, 10.0
FORCE_RECOMPUTE_FEATURES = False

MIN_WORD_COUNT, TOP_K_WORDS = 15, 10
LAMBDA_VALUES = [0.0, 0.01, 0.1, 0.5, 1.0]
N_BOOTSTRAPS = 100 if RUN_MODE == "smoke" else 500
COUNTERFACTUAL_TRAIN_PAIRS = {"smoke": 30, "standard": 120, "extended": 250}[RUN_MODE]

PROJECT_DIR = Path.cwd()
CACHE_DIR = PROJECT_DIR / "cache"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
for directory in (CACHE_DIR, RESULTS_DIR, FIGURE_DIR): directory.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
try: torch.use_deterministic_algorithms(True, warn_only=True)
except Exception: pass
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 2**30
except Exception: ram_gb = float("nan")
disk_gb = shutil.disk_usage(PROJECT_DIR).free / 2**30
print(f"Runtime device: {device}\nRAM: {ram_gb:.1f} GiB\nAvailable disk: {disk_gb:.1f} GiB")
print(f"Mode: {RUN_MODE}; target pairs: {MAX_PAIRS:,}")


## 3. Stream AMI and inspect the live schema

The dataset is never materialized in full. The first pass removes the audio column and reads only pairing metadata. A
later pass revisits the stream and decodes only selected audio IDs. The schema check below intentionally inspects each
configuration independently; pairing never assumes common row order.


In [ ]:
DATASET_NAME = "edinburghcstr/ami"
REQUIRED_COLUMNS = {"meeting_id","audio_id","text","audio","begin_time","end_time","microphone_id","speaker_id"}

def open_ami(config, include_audio=False):
    ds = load_dataset(DATASET_NAME, config, split=AMI_SPLIT, streaming=True)
    missing = REQUIRED_COLUMNS - set(ds.column_names)
    if missing: raise RuntimeError(f"AMI/{config} is missing required columns: {sorted(missing)}")
    try:
        ds = ds.cast_column("audio", HFAudio(sampling_rate=16000, decode=False))
    except Exception as cast_exc:
        try:
            ds = ds.decode(False)
        except Exception as decode_exc:
            raise RuntimeError(
                "Unable to disable Hugging Face audio decoding; update `datasets` or install a compatible version. "
                f"cast error={cast_exc}; decode error={decode_exc}"
            ) from decode_exc
    return ds if include_audio else ds.remove_columns(["audio"])

for config in ("ihm", "sdm"):
    sample = next(iter(open_ami(config, include_audio=False)))
    print(config.upper(), {k: (type(v).__name__, str(v)[:70]) for k,v in sample.items()})


## 4. Build verified IHM/SDM pairs

Rows are matched by meeting, speaker, normalized transcript, and begin/end times within 20 ms. This tolerates float32
rounding while remaining strict enough to reject unrelated speech. A short SHA-1 digest names the verified event. Empty,
out-of-range, duplicate, and timestamp-mismatched items are counted rather than silently discarded.


In [ ]:
def normalize_text(text):
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9']+", " ", str(text).upper())).strip()

def eligible_metadata(row):
    try:
        duration = float(row["end_time"]) - float(row["begin_time"])
        return (MIN_DURATION <= duration <= MAX_DURATION and bool(normalize_text(row["text"]))
                and bool(row["meeting_id"]) and bool(row["speaker_id"]) and bool(row["audio_id"]))
    except Exception:
        return False

def coarse_key(row):
    return (str(row["meeting_id"]), str(row["speaker_id"]), normalize_text(row["text"]))

def exactish_time(row):
    return (round(float(row["begin_time"])*100), round(float(row["end_time"])*100))

IHM_SCAN_CAP = min(14000, max(4000, 4*MAX_PAIRS if RUN_MODE == "smoke" else 3*MAX_PAIRS))
SDM_SCAN_CAP = 15000
exclusions = []
ihm_index = defaultdict(list)
for i, row in enumerate(tqdm(open_ami("ihm"), total=IHM_SCAN_CAP, desc="IHM metadata")):
    if i >= IHM_SCAN_CAP: break
    if eligible_metadata(row):
        ihm_index[coarse_key(row)].append({k: row[k] for k in row})
    else: exclusions.append({"config":"ihm","audio_id":row.get("audio_id"),"reason":"metadata_filter"})

used_ihm, pairs = set(), []
for i, sdm in enumerate(tqdm(open_ami("sdm"), total=SDM_SCAN_CAP, desc="SDM matching")):
    if i >= SDM_SCAN_CAP or len(pairs) >= MAX_PAIRS: break
    if not eligible_metadata(sdm):
        exclusions.append({"config":"sdm","audio_id":sdm.get("audio_id"),"reason":"metadata_filter"}); continue
    sb,se = exactish_time(sdm); candidates=[]
    for ihm in ihm_index.get(coarse_key(sdm), []):
        if ihm["audio_id"] in used_ihm: continue
        ib,ie = exactish_time(ihm)
        if abs(sb-ib) <= 2 and abs(se-ie) <= 2:
            candidates.append((abs(sb-ib)+abs(se-ie), ihm))
    if not candidates: continue
    _, ihm = min(candidates, key=lambda item: item[0]); used_ihm.add(ihm["audio_id"])
    signature = "|".join(map(str, (*coarse_key(ihm), *exactish_time(ihm))))
    pairs.append({
        "pair_id": hashlib.sha1(signature.encode()).hexdigest()[:16],
        "meeting_id": ihm["meeting_id"], "speaker_id": ihm["speaker_id"], "text": ihm["text"],
        "normalized_text": normalize_text(ihm["text"]), "begin_time": float(ihm["begin_time"]),
        "end_time": float(ihm["end_time"]), "duration": float(ihm["end_time"])-float(ihm["begin_time"]),
        "ihm_audio_id": ihm["audio_id"], "sdm_audio_id": sdm["audio_id"],
        "ihm_microphone_id": ihm["microphone_id"], "sdm_microphone_id": sdm["microphone_id"],
        "time_error_ms": 10*(abs(sb-exactish_time(ihm)[0])+abs(se-exactish_time(ihm)[1]))})

pair_df = pd.DataFrame(pairs)
if pair_df.empty: raise RuntimeError("No AMI pairs found. Inspect the schema samples and Hugging Face status.")
if pair_df.pair_id.duplicated().any(): raise AssertionError("Duplicate pair IDs after matching")
if len(pair_df) < MAX_PAIRS: warnings.warn(f"Found {len(pair_df)} pairs, below requested {MAX_PAIRS}; continuing safely.")
pair_df.to_csv(RESULTS_DIR/"paired_manifest.csv", index=False)
excluded_df = pd.DataFrame(exclusions)
excluded_df.to_csv(RESULTS_DIR/"excluded_samples.csv", index=False)
print(f"Matched {len(pair_df):,} verified pairs; metadata exclusions: {len(excluded_df):,}")
print(f"Maximum accepted timestamp error: {pair_df.time_error_ms.max():.0f} ms")


## 5. Dataset audit

The audit checks sample balance before modeling. Speaker and meeting concentration matter because a classifier can otherwise
exploit dominant participants or sessions. The tables remain compact; the full paired manifest is exported.


In [ ]:
summary = pd.DataFrame([{
    "run_mode":RUN_MODE, "matched_pairs":len(pair_df), "speakers":pair_df.speaker_id.nunique(),
    "meetings":pair_df.meeting_id.nunique(), "median_duration_s":pair_df.duration.median(),
    "total_paired_audio_hours":2*pair_df.duration.sum()/3600, "unique_transcripts":pair_df.normalized_text.nunique(),
    "metadata_exclusions":len(excluded_df)}])
summary.to_csv(RESULTS_DIR/"dataset_summary.csv",index=False); display(summary)
print("IHM microphones:", pair_df.ihm_microphone_id.value_counts().to_dict())
print("SDM microphones:", pair_df.sdm_microphone_id.value_counts().to_dict())
display(pair_df.sample(min(8,len(pair_df)),random_state=SEED)[["speaker_id","meeting_id","text","duration"]])

fig,axes=plt.subplots(1,3,figsize=(16,4.2))
axes[0].hist(pair_df.duration,bins=30); axes[0].set(xlabel="Duration (s)",ylabel="Pairs",title="Utterance duration")
pair_df.speaker_id.value_counts().head(20).sort_values().plot.barh(ax=axes[1]); axes[1].set(xlabel="Pairs",title="Top speakers")
pair_df.meeting_id.value_counts().head(20).sort_values().plot.barh(ax=axes[2]); axes[2].set(xlabel="Pairs",title="Top meetings")
fig.suptitle("AMI paired-subset composition"); plt.tight_layout(); fig.savefig(FIGURE_DIR/"dataset_audit.png",dpi=200); plt.show()
dominant = pair_df.speaker_id.value_counts(normalize=True).iloc[0]
print(f"Largest speaker share: {dominant:.1%}" + (" — interpret with care." if dominant>.2 else "."))


## 6. Defensive selective audio access

`decode=False` avoids changing `datasets`/`torchcodec` APIs. The helper accepts encoded bytes, local paths, URLs, `hf://`
paths, or a newer decoder object. Failed records are collected. Only requested IDs are decoded and each waveform is
converted to mono 16 kHz float32.


In [ ]:
def decode_audio_object(obj, target_sr=16000):
    if hasattr(obj,"get_all_samples"):
        samples=obj.get_all_samples(); array=np.asarray(samples.data).squeeze(); sr=int(samples.sample_rate)
    elif isinstance(obj,dict):
        if obj.get("array") is not None:
            array=np.asarray(obj["array"]); sr=int(obj.get("sampling_rate",target_sr))
        elif obj.get("bytes") is not None:
            array,sr=sf.read(io.BytesIO(obj["bytes"]),dtype="float32",always_2d=False)
        elif obj.get("path"):
            with fsspec.open(obj["path"],"rb") as handle: array,sr=sf.read(handle,dtype="float32",always_2d=False)
        else: raise ValueError("Audio dictionary has no array, bytes, or path")
    elif isinstance(obj,(str,Path)):
        with fsspec.open(str(obj),"rb") as handle: array,sr=sf.read(handle,dtype="float32",always_2d=False)
    else: raise TypeError(f"Unsupported audio object: {type(obj)}")
    array=np.asarray(array,dtype=np.float32)
    if array.ndim>1: array=array.mean(axis=0 if array.shape[0]<array.shape[1] else 1)
    if sr!=target_sr:
        divisor=np.gcd(sr,target_sr); array=signal.resample_poly(array,target_sr//divisor,sr//divisor).astype(np.float32); sr=target_sr
    if not np.isfinite(array).all() or len(array)<target_sr*MIN_DURATION/2: raise ValueError("Malformed or too-short waveform")
    return array,sr

audio_failures=[]
def stream_selected(config, audio_id_to_pair):
    remaining=set(audio_id_to_pair)
    stream=open_ami(config,include_audio=True)
    for row in tqdm(stream,desc=f"{config.upper()} selective audio"):
        audio_id=row.get("audio_id")
        if audio_id not in remaining: continue
        try:
            audio,sr=decode_audio_object(row["audio"])
            yield audio_id,audio,sr
        except Exception as exc:
            audio_failures.append({"config":config,"audio_id":audio_id,"reason":f"{type(exc).__name__}: {exc}"})
        remaining.remove(audio_id)
        if not remaining: break
    for audio_id in sorted(remaining): audio_failures.append({"config":config,"audio_id":audio_id,"reason":"not_found_in_stream"})

demo_pairs=pair_df.sample(min(3,len(pair_df)),random_state=SEED)
demo_audio={}
for config,column in (("ihm","ihm_audio_id"),("sdm","sdm_audio_id")):
    wanted=dict(zip(demo_pairs[column],demo_pairs.pair_id))
    for audio_id,audio,sr in stream_selected(config,wanted): demo_audio[(wanted[audio_id],config)]=(audio,sr)
print("Decoded demo recordings:",len(demo_audio),"of",2*len(demo_pairs))


## 7. Listen to paired examples and inspect signal differences

These widgets make the recording-condition contrast tangible. The plots show waveforms, log-mel spectrograms, average
spectra, and the SDM−IHM log-spectral difference. Amplitude and timing differences are left visible rather than normalized
away.


In [ ]:
for row in demo_pairs.itertuples():
    print(f"{row.speaker_id} | {row.meeting_id} | {row.duration:.2f}s\n{row.text}\nIHM")
    if (row.pair_id,"ihm") in demo_audio: display(Audio(demo_audio[(row.pair_id,"ihm")][0],rate=16000))
    print("SDM")
    if (row.pair_id,"sdm") in demo_audio: display(Audio(demo_audio[(row.pair_id,"sdm")][0],rate=16000))


In [ ]:
def logmel(y,sr=16000):
    mel=librosa.feature.melspectrogram(y=y,sr=sr,n_fft=512,hop_length=160,n_mels=64,power=2)
    return librosa.power_to_db(mel,ref=np.max)

for row in demo_pairs.head(2).itertuples():
    if any((row.pair_id,c) not in demo_audio for c in ("ihm","sdm")): continue
    h,_=demo_audio[(row.pair_id,"ihm")]; d,_=demo_audio[(row.pair_id,"sdm")]
    fig,axes=plt.subplots(2,2,figsize=(13,7))
    for ax,y,label in ((axes[0,0],h,"IHM"),(axes[0,1],d,"SDM")):
        ax.plot(np.arange(len(y))/16000,y,lw=.5); ax.set(title=f"{label} waveform",xlabel="Time (s)",ylabel="Amplitude")
    for ax,y,label in ((axes[1,0],h,"IHM"),(axes[1,1],d,"SDM")):
        L=logmel(y); extent=[0,len(y)/16000,0,8000]; im=ax.imshow(L,origin="lower",aspect="auto",extent=extent,cmap="magma")
        ax.set(title=f"{label} log-mel",xlabel="Time (s)",ylabel="Frequency (Hz)"); fig.colorbar(im,ax=ax,label="dB")
    fig.suptitle(f"Matched event: {row.meeting_id}, {row.speaker_id}"); plt.tight_layout();
    fig.savefig(FIGURE_DIR/f"pair_{row.pair_id}_wave_mel.png",dpi=200); plt.show()
    f,ph=signal.welch(h,16000,nperseg=1024); _,pd_=signal.welch(d,16000,nperseg=1024)
    hd=10*np.log10(ph+1e-12); dd=10*np.log10(pd_+1e-12)
    fig,axes=plt.subplots(1,2,figsize=(12,4)); axes[0].plot(f,hd,label="IHM"); axes[0].plot(f,dd,label="SDM")
    axes[0].set(xlabel="Frequency (Hz)",ylabel="PSD (dB/Hz)",title="Average spectra"); axes[0].legend()
    axes[1].plot(f,dd-hd); axes[1].axhline(0,color="black",lw=.7); axes[1].set(xlabel="Frequency (Hz)",ylabel="SDM − IHM (dB)",title="Paired spectral difference")
    plt.tight_layout(); fig.savefig(FIGURE_DIR/f"pair_{row.pair_id}_spectra.png",dpi=200); plt.show()


## 8. Extract and cache interpretable acoustic features

Each recording becomes a fixed vector containing MFCC, delta, and delta-delta means/standard deviations; spectral shape,
energy, and zero-crossing summaries; a fast YIN pitch summary; voiced fraction; and duration. CMN and CMVN are applied to
the MFCC frame sequence before aggregation. Metadata columns are kept separately and are never model inputs.

The stream is traversed once per microphone condition. Audio arrays are released immediately after feature extraction.


In [ ]:
N_MFCC, N_FFT, HOP = 13, 512, 160
STAT_FEATURES = ["centroid","bandwidth","rolloff","flatness","rms","zcr"] + [f"contrast_{i}" for i in range(7)]

def summarize_frames(x):
    x=np.asarray(x,float); x=np.atleast_2d(x)
    return np.r_[np.nanmean(x,axis=1),np.nanstd(x,axis=1)]

def acoustic_variants(y,sr=16000):
    if len(y)<N_FFT: y=np.pad(y,(0,N_FFT-len(y)))
    S=np.abs(librosa.stft(y,n_fft=N_FFT,hop_length=HOP,win_length=400)); power=S**2
    log_mel=librosa.power_to_db(librosa.feature.melspectrogram(S=power,sr=sr,n_mels=40)+1e-10)
    base_mfcc=librosa.feature.mfcc(S=log_mel,n_mfcc=N_MFCC)
    spectral={
        "centroid":librosa.feature.spectral_centroid(S=S,sr=sr),
        "bandwidth":librosa.feature.spectral_bandwidth(S=S,sr=sr),
        "rolloff":librosa.feature.spectral_rolloff(S=S,sr=sr),
        "flatness":librosa.feature.spectral_flatness(S=S),
        "rms":librosa.feature.rms(S=S,frame_length=N_FFT),
        "zcr":librosa.feature.zero_crossing_rate(y,frame_length=N_FFT,hop_length=HOP),
        "contrast":librosa.feature.spectral_contrast(S=S,sr=sr),}
    rms=spectral["rms"].ravel(); threshold=max(1e-5,np.percentile(rms,30)*1.5)
    try:
        f0=librosa.yin(y,fmin=60,fmax=450,sr=sr,frame_length=1024,hop_length=HOP)
        n=min(len(f0),len(rms)); voiced=rms[:n]>threshold; f0=np.asarray(f0[:n]); f0[~voiced]=np.nan
        finite=f0[np.isfinite(f0)]; pitch=[np.median(finite),np.mean(finite),np.std(finite),np.mean(voiced)] if finite.size else [0,0,0,0]
    except Exception: pitch=[0,0,0,0]
    spectral_matrix=np.vstack([spectral[k] for k in ("centroid","bandwidth","rolloff","flatness","rms","zcr")] + [spectral["contrast"]])
    tail=np.r_[summarize_frames(spectral_matrix),pitch,len(y)/sr]
    variants={}
    for name,mfcc in {
        "raw":base_mfcc,
        "cmn":base_mfcc-base_mfcc.mean(axis=1,keepdims=True),
        "cmvn":(base_mfcc-base_mfcc.mean(axis=1,keepdims=True))/(base_mfcc.std(axis=1,keepdims=True)+1e-8),}.items():
        width=min(9,mfcc.shape[1] if mfcc.shape[1]%2 else mfcc.shape[1]-1)
        if width>=3:
            delta=librosa.feature.delta(mfcc,width=width,mode="nearest"); delta2=librosa.feature.delta(mfcc,width=width,order=2,mode="nearest")
        else: delta=np.zeros_like(mfcc); delta2=np.zeros_like(mfcc)
        variants[name]=np.asarray(np.r_[summarize_frames(np.vstack([mfcc,delta,delta2])),tail],dtype=np.float32)
    return variants

feature_names=[]
for stat in ("mean","std"):
    for group in ("mfcc","delta","delta2"):
        feature_names += [f"{group}_{i:02d}_{stat}" for i in range(N_MFCC)]
for stat in ("mean","std"): feature_names += [f"{name}_{stat}" for name in STAT_FEATURES]
feature_names += ["f0_median","f0_mean","f0_std","voiced_fraction","duration_feature"]
assert len(feature_names)==len(acoustic_variants(np.zeros(16000))["raw"])

pair_digest=hashlib.sha1("|".join(sorted(pair_df.pair_id)).encode()).hexdigest()[:12]
cache_signature=hashlib.sha1(json.dumps({"dataset":DATASET_NAME,"split":AMI_SPLIT,"mode":RUN_MODE,"pair_digest":pair_digest,
    "min_d":MIN_DURATION,"max_d":MAX_DURATION,"feature_version":3},sort_keys=True).encode()).hexdigest()[:12]
feature_cache=CACHE_DIR/f"ami_features_{RUN_MODE}_{cache_signature}.parquet"

if feature_cache.exists() and not FORCE_RECOMPUTE_FEATURES:
    feature_df=pd.read_parquet(feature_cache); print("Loaded",feature_cache)
else:
    feature_rows=[]
    meta_by_id={}
    for row in pair_df.itertuples():
        meta_by_id[row.ihm_audio_id]=(row.pair_id,"IHM",row)
        meta_by_id[row.sdm_audio_id]=(row.pair_id,"SDM",row)
    for config,column in (("ihm","ihm_audio_id"),("sdm","sdm_audio_id")):
        wanted={getattr(row,column):row.pair_id for row in pair_df.itertuples()}
        for audio_id,y,sr in stream_selected(config,wanted):
            pair_id,channel,row=meta_by_id[audio_id]
            try:
                if abs(len(y)/sr-row.duration) > 0.35:
                    raise ValueError(f"decoded duration {len(y)/sr:.3f}s differs from metadata {row.duration:.3f}s")
                variants=acoustic_variants(y,sr)
                for representation,vector in variants.items():
                    record={"pair_id":pair_id,"channel":channel,"representation":representation,
                            "speaker_id":row.speaker_id,"meeting_id":row.meeting_id,"text":row.text,
                            "normalized_text":row.normalized_text,"audio_id":audio_id}
                    record.update(dict(zip(feature_names,vector))); feature_rows.append(record)
            except Exception as exc:
                audio_failures.append({"config":config,"audio_id":audio_id,"reason":f"feature_error: {type(exc).__name__}: {exc}"})
            del y; gc.collect()
    if not feature_rows:
        pd.DataFrame(audio_failures).to_csv(RESULTS_DIR/"audio_failures.csv",index=False)
        raise RuntimeError("No audio features were extracted. Inspect audio_failures.csv and rerun after fixing decoding/network access.")
    feature_df=pd.DataFrame(feature_rows)
    complete=feature_df.groupby(["pair_id","representation"]).channel.nunique().unstack(fill_value=0)
    required=["raw","cmn","cmvn"]
    if not set(required).issubset(complete.columns): raise RuntimeError("One or more required feature representations are missing.")
    valid_pairs=set(complete.index[complete[required].eq(2).all(axis=1)])
    feature_df=feature_df[feature_df.pair_id.isin(valid_pairs)].copy()
    if feature_df.empty: raise RuntimeError("No pairs have both channels for every required feature representation.")
    feature_df.to_parquet(feature_cache,index=False)
feature_df.to_parquet(RESULTS_DIR/"features.parquet",index=False)
pd.DataFrame(audio_failures,columns=["config","audio_id","reason"]).to_csv(RESULTS_DIR/"audio_failures.csv",index=False)
print(f"Feature rows: {len(feature_df):,}; complete pairs: {feature_df.pair_id.nunique():,}; dimension: {len(feature_names)}")
print("Audio exclusions:",len(audio_failures),Counter(x["reason"].split(":")[0] for x in audio_failures))


## 9. Paired feature movement before modeling

Matched IHM↔SDM distances are compared with randomly mismatched cross-channel distances. Matched pairs share speech
content; unmatched pairs generally change both content and recording condition, so this is descriptive rather than a
causal decomposition.


In [ ]:
raw=feature_df[feature_df.representation=="raw"].copy()
h=raw[raw.channel=="IHM"].set_index("pair_id"); d=raw[raw.channel=="SDM"].set_index("pair_id")
common=h.index.intersection(d.index); H=h.loc[common,feature_names].to_numpy(); D=d.loc[common,feature_names].to_numpy()
scaler_distance=StandardScaler().fit(np.vstack([H,D])); Hs=scaler_distance.transform(H); Ds=scaler_distance.transform(D)
rng=np.random.default_rng(SEED); permutation=rng.permutation(len(common))
mfcc_idx=[i for i,n in enumerate(feature_names) if n.startswith(("mfcc_","delta_","delta2_"))]
spectral_idx=[i for i in range(len(feature_names)) if i not in mfcc_idx]
cos=lambda a,b:1-np.sum(a*b,axis=1)/(np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1)+1e-9)
distances=pd.DataFrame({
    "matched_euclidean":np.linalg.norm(Hs-Ds,axis=1),"unmatched_euclidean":np.linalg.norm(Hs-Ds[permutation],axis=1),
    "matched_cosine":cos(Hs,Ds),"unmatched_cosine":cos(Hs,Ds[permutation]),
    "matched_mfcc":np.linalg.norm(Hs[:,mfcc_idx]-Ds[:,mfcc_idx],axis=1),
    "matched_spectral":np.linalg.norm(Hs[:,spectral_idx]-Ds[:,spectral_idx],axis=1)})
display(distances.describe().loc[["mean","50%","std"]])
fig,axes=plt.subplots(1,2,figsize=(12,4.2))
axes[0].hist(distances.matched_euclidean,bins=35,alpha=.65,label="matched"); axes[0].hist(distances.unmatched_euclidean,bins=35,alpha=.65,label="unmatched")
axes[0].set(xlabel="Standardized Euclidean distance",ylabel="Pairs",title="Cross-channel feature distance"); axes[0].legend()
axes[1].hist(distances.matched_cosine,bins=35,alpha=.65,label="matched"); axes[1].hist(distances.unmatched_cosine,bins=35,alpha=.65,label="unmatched")
axes[1].set(xlabel="Cosine distance",ylabel="Pairs",title="Angular feature distance"); axes[1].legend()
plt.tight_layout(); fig.savefig(FIGURE_DIR/"matched_unmatched_distances.png",dpi=200); plt.show()


## 10. Leakage-safe primary split

The primary split groups by speaker, then assigns every pair (and both channels) together. Scalers, PCA, probes, and
counterfactual response estimates are fit only from training rows. The assertions are executable safeguards, not merely
documentation.


In [ ]:
pair_meta=pair_df[pair_df.pair_id.isin(feature_df.pair_id.unique())].reset_index(drop=True)
speakers=pair_meta.speaker_id.nunique()
if speakers<6: raise RuntimeError(f"Only {speakers} speakers available; grouped train/validation/test evaluation is unstable.")
outer=GroupShuffleSplit(n_splits=1,train_size=.70,random_state=SEED)
train_i,rest_i=next(outer.split(pair_meta,groups=pair_meta.speaker_id))
rest=pair_meta.iloc[rest_i].reset_index(drop=True)
inner=GroupShuffleSplit(n_splits=1,train_size=.50,random_state=SEED+1)
val_j,test_j=next(inner.split(rest,groups=rest.speaker_id))
split_map={p:"train" for p in pair_meta.iloc[train_i].pair_id}
split_map.update({p:"validation" for p in rest.iloc[val_j].pair_id}); split_map.update({p:"test" for p in rest.iloc[test_j].pair_id})
pair_meta["split"]=pair_meta.pair_id.map(split_map); feature_df["split"]=feature_df.pair_id.map(split_map)
train_s=set(pair_meta.loc[pair_meta.split=="train","speaker_id"]); val_s=set(pair_meta.loc[pair_meta.split=="validation","speaker_id"]); test_s=set(pair_meta.loc[pair_meta.split=="test","speaker_id"])
assert train_s.isdisjoint(val_s|test_s) and val_s.isdisjoint(test_s)
assert set(pair_meta.loc[pair_meta.split=="train","pair_id"]).isdisjoint(pair_meta.loc[pair_meta.split=="test","pair_id"])
assert feature_df.groupby("pair_id").split.nunique().max()==1
print({part:{"speakers":pair_meta.loc[pair_meta.split==part,"speaker_id"].nunique(),"pairs":(pair_meta.split==part).sum()} for part in ("train","validation","test")})


## 11. Experiment 1 — can we predict the microphone?

**Question:** do acoustic features reveal IHM versus SDM for speakers never seen during training?  
**Control:** the test speakers and pair IDs are disjoint from training; scaling uses training rows only.  
**Evidence of sensitivity:** balanced accuracy and macro-F1 well above the 50% chance baseline.


In [ ]:
def metrics_dict(y_true,y_pred,score=None):
    result={"accuracy":accuracy_score(y_true,y_pred),"balanced_accuracy":balanced_accuracy_score(y_true,y_pred),"macro_f1":f1_score(y_true,y_pred,average="macro")}
    try: result["roc_auc"]=roc_auc_score(y_true,score) if score is not None and len(np.unique(y_true))==2 else np.nan
    except ValueError: result["roc_auc"]=np.nan
    return result

def make_classifier(name,scaler="standard"):
    scale=StandardScaler() if scaler=="standard" else RobustScaler() if scaler=="robust" else "passthrough"
    if name=="Logistic Regression": clf=LogisticRegression(max_iter=2500,class_weight="balanced",random_state=SEED)
    elif name=="Linear SVM": clf=SVC(kernel="linear",probability=True,class_weight="balanced",random_state=SEED)
    elif name=="Random Forest": clf=RandomForestClassifier(n_estimators=300,class_weight="balanced",n_jobs=-1,random_state=SEED)
    else: raise ValueError(name)
    return Pipeline([("scale",scale),("classifier",clf)])

raw=feature_df[feature_df.representation=="raw"].copy(); raw["mic_target"]=(raw.channel=="SDM").astype(int)
train=raw.split=="train"; test=raw.split=="test"; Xtr=raw.loc[train,feature_names]; Xte=raw.loc[test,feature_names]; ytr=raw.loc[train,"mic_target"]; yte=raw.loc[test,"mic_target"]
mic_rows=[]; mic_models={}; prediction_registry={}
validation=raw.split=="validation"; Xval=raw.loc[validation,feature_names]; yval=raw.loc[validation,"mic_target"]
for name in ("Logistic Regression","Linear SVM","Random Forest"):
    model=make_classifier(name); model.fit(Xtr,ytr)
    val_pred=model.predict(Xval); val_score=model.predict_proba(Xval)[:,1]
    pred=model.predict(Xte); score=model.predict_proba(Xte)[:,1]
    result={"experiment":"microphone","model":name,"representation":"Raw",
            "validation_balanced_accuracy":balanced_accuracy_score(yval,val_pred),**metrics_dict(yte,pred,score)}
    mic_rows.append(result); mic_models[name]=model; prediction_registry[f"mic_{name}"]=(yte.to_numpy(),pred)
mic_results=pd.DataFrame(mic_rows).sort_values("validation_balanced_accuracy",ascending=False); display(mic_results)
mic_results.to_csv(RESULTS_DIR/"baseline_results.csv",index=False)
best_mic_name=mic_results.iloc[0].model; best_mic_model=mic_models[best_mic_name]
ConfusionMatrixDisplay.from_predictions(yte,best_mic_model.predict(Xte),display_labels=["IHM","SDM"],cmap="Blues",colorbar=False)
plt.title(f"{best_mic_name}: held-out speakers"); plt.tight_layout(); plt.savefig(FIGURE_DIR/"microphone_confusion.png",dpi=200); plt.show()

logistic=mic_models["Logistic Regression"]; coefficients=logistic.named_steps["classifier"].coef_[0]
important=np.argsort(np.abs(coefficients))[-20:]
fig,ax=plt.subplots(figsize=(8,6)); colors=plt.cm.coolwarm((coefficients[important]-coefficients[important].min())/(np.ptp(coefficients[important])+1e-9))
ax.barh(np.array(feature_names)[important],coefficients[important],color=colors); ax.axvline(0,color="black",lw=.8)
ax.set(xlabel="Standardized logistic coefficient (positive = SDM)",title="Features most associated with recording condition")
plt.tight_layout(); fig.savefig(FIGURE_DIR/"microphone_coefficients.png",dpi=200); plt.show()
score=mic_results.loc[mic_results.model=="Logistic Regression","balanced_accuracy"].iloc[0]
print(f"Speaker-independent microphone balanced accuracy: {score:.1%}. " + ("The representation contains strong condition information." if score>=.70 else "Evidence of channel separation is modest." if score>=.58 else "Performance is near chance."))


## 12. Experiment 2 — does speaker information survive a channel change?

A literal speaker-ID classifier cannot be tested on speakers absent from training: every test label would be a new class.
Therefore this is explicitly a **closed-set speaker-information probe**, not the primary speaker-independent benchmark.
Speakers occur on both sides, but utterance pairs are disjoint and both channels of each event remain together. This
answers whether a representation retains identity information across channels without pretending to measure unseen-speaker
classification. The microphone experiment above and lexical experiment below remain speaker-independent.


In [ ]:
pair_counts=pair_meta.groupby("speaker_id").pair_id.nunique().sort_values(ascending=False)
thresholds=[20,12,8,6]
eligible_speakers=[]
for threshold in thresholds:
    eligible_speakers=pair_counts[pair_counts>=threshold].head(12).index.tolist()
    if len(eligible_speakers)>=4: break
speaker_probe_available=len(eligible_speakers)>=4
speaker_split_map={}
if speaker_probe_available:
    rng=np.random.default_rng(SEED)
    for speaker in eligible_speakers:
        ids=pair_meta.loc[pair_meta.speaker_id==speaker,"pair_id"].to_numpy().copy(); rng.shuffle(ids); n=len(ids)
        n_train=max(2,int(.70*n)); n_val=max(1,int(.15*n));
        if n_train+n_val>=n: n_train,n_val=n-2,1
        speaker_split_map.update({p:"train" for p in ids[:n_train]}); speaker_split_map.update({p:"validation" for p in ids[n_train:n_train+n_val]}); speaker_split_map.update({p:"test" for p in ids[n_train+n_val:]})
    speaker_pairs=set(speaker_split_map); assert set(p for p,s in speaker_split_map.items() if s=="train").isdisjoint(p for p,s in speaker_split_map.items() if s=="test")
    print(f"Closed-set speaker probe: {len(eligible_speakers)} speakers, {len(speaker_pairs)} pairs")
else: warnings.warn("Too few well-represented speakers; speaker probe and invariant task will be skipped.")

def domain_probe(frame,label_col,split_column,source,target,model=None):
    tr=(frame[split_column]=="train") & (frame.channel==source); te=(frame[split_column]=="test") & (frame.channel==target)
    labels=set(frame.loc[tr,label_col]) & set(frame.loc[te,label_col]); tr &= frame[label_col].isin(labels); te &= frame[label_col].isin(labels)
    if len(labels)<2 or tr.sum()<10 or te.sum()<4: return None,None,None
    model=clone(model or make_classifier("Logistic Regression")); model.fit(frame.loc[tr,feature_names],frame.loc[tr,label_col]); pred=model.predict(frame.loc[te,feature_names])
    return metrics_dict(frame.loc[te,label_col],pred),model,(frame.loc[te,label_col].to_numpy(),pred)

speaker_results=[]; speaker_models={}
if speaker_probe_available:
    speaker_raw=raw[raw.pair_id.isin(speaker_pairs)].copy(); speaker_raw["speaker_probe_split"]=speaker_raw.pair_id.map(speaker_split_map)
    for source,target in (("IHM","IHM"),("SDM","SDM"),("IHM","SDM"),("SDM","IHM")):
        met,model,pred=domain_probe(speaker_raw,"speaker_id","speaker_probe_split",source,target)
        if met:
            condition=f"{source}→{target}"; speaker_results.append({"condition":condition,**met}); speaker_models[condition]=model; prediction_registry[f"speaker_{condition}"]=pred
speaker_results=pd.DataFrame(speaker_results); display(speaker_results)
if not speaker_results.empty:
    fig,ax=plt.subplots(figsize=(7,4.5)); ax.bar(speaker_results.condition,speaker_results.macro_f1)
    ax.set(ylim=(0,1),ylabel="Macro-F1",title="Closed-set speaker information across channels"); plt.tight_layout(); fig.savefig(FIGURE_DIR/"speaker_cross_channel.png",dpi=200); plt.show()
    same=speaker_results[speaker_results.condition.isin(["IHM→IHM","SDM→SDM"])].macro_f1.mean(); cross=speaker_results[speaker_results.condition.isin(["IHM→SDM","SDM→IHM"])].macro_f1.mean()
    print(f"Mean same-channel speaker F1={same:.3f}; cross-channel F1={cross:.3f}; change={cross-same:+.3f}.")


## 13. Experiment 3 — lexical information across channels

Only single-word transcripts with enough pairs and at least three speakers are considered. The train/test speakers remain
disjoint. If the bounded subset cannot support a defensible task, the notebook records `NaN` and explains the skip rather
than inventing a result.


In [ ]:
pair_meta["word_label"]=pair_meta.normalized_text.str.lower()
training_meta=pair_meta[pair_meta.split=="train"].copy()
single=training_meta.word_label.str.match(r"^[a-z']+$",na=False)
word_stats=training_meta[single].groupby("word_label").agg(pairs=("pair_id","nunique"),speakers=("speaker_id","nunique"))
word_labels=[]; used_word_threshold=None
for threshold in [MIN_WORD_COUNT,10,5]:
    candidates=word_stats[(word_stats.pairs>=threshold)&(word_stats.speakers>=3)].sort_values("pairs",ascending=False).head(TOP_K_WORDS).index.tolist()
    if len(candidates)>=2: word_labels=candidates; used_word_threshold=threshold; break
lexical_available=len(word_labels)>=2
lexical_results=[]; lexical_models={}
if lexical_available:
    lexical_raw=raw.merge(pair_meta[["pair_id","word_label"]],on="pair_id",how="left",suffixes=("","_pair")); lexical_raw=lexical_raw[lexical_raw.word_label.isin(word_labels)]
    print(f"Lexical task: {len(word_labels)} words at minimum count {used_word_threshold}: {word_labels}")
    for source,target in (("IHM","IHM"),("SDM","SDM"),("IHM","SDM"),("SDM","IHM")):
        met,model,pred=domain_probe(lexical_raw,"word_label","split",source,target)
        if met:
            condition=f"{source}→{target}"; lexical_results.append({"condition":condition,**met}); lexical_models[condition]=model; prediction_registry[f"lexical_{condition}"]=pred
    lexical_results=pd.DataFrame(lexical_results); display(lexical_results)
    if not lexical_results.empty:
        fig,ax=plt.subplots(figsize=(7,4.5)); ax.bar(lexical_results.condition,lexical_results.macro_f1)
        ax.set(ylim=(0,1),ylabel="Macro-F1",title="Speaker-independent lexical classification"); plt.tight_layout(); fig.savefig(FIGURE_DIR/"lexical_cross_channel.png",dpi=200); plt.show()
else:
    lexical_results=pd.DataFrame(); lexical_raw=pd.DataFrame(); print("Lexical experiment skipped: fewer than two viable single-word classes across disjoint speaker splits.")


## 14. Channel normalization tradeoff

For each representation we re-fit preprocessing on training rows only. “Raw” still uses a train-fitted standard scaler
inside the linear classifier, as required for numerical comparability. CMN/CMVN operate at the frame level. Robust scaling
and PCA whitening are fit on train data. CORAL is evaluated separately for cross-channel speaker transfer and is explicitly
an **unsupervised domain-adaptation** result because it sees unlabeled target-channel recordings from training pairs.


In [ ]:
def pipeline_for(method):
    if method in ("Raw","CMN","CMVN"): prep=StandardScaler()
    elif method=="Robust": prep=RobustScaler()
    elif method=="PCA whiten": prep=Pipeline([("scale",StandardScaler()),("pca",PCA(n_components=.95,whiten=True,random_state=SEED))])
    else: raise ValueError(method)
    return Pipeline([("preprocess",prep),("classifier",LogisticRegression(max_iter=2500,class_weight="balanced",random_state=SEED))])

def eval_mic_method(method,representation):
    frame=feature_df[feature_df.representation==representation].copy(); frame["target"]=(frame.channel=="SDM").astype(int)
    tr=frame.split=="train"; va=frame.split=="validation"; te=frame.split=="test"; model=pipeline_for(method); model.fit(frame.loc[tr,feature_names],frame.loc[tr,"target"])
    val_pred=model.predict(frame.loc[va,feature_names]); pred=model.predict(frame.loc[te,feature_names]); score=model.predict_proba(frame.loc[te,feature_names])[:,1]
    return metrics_dict(frame.loc[te,"target"],pred,score),balanced_accuracy_score(frame.loc[va,"target"],val_pred),model

def eval_task_method(method,representation,task):
    base=feature_df[feature_df.representation==representation].copy(); base["split"]=base.pair_id.map(split_map)
    label_col="word_label" if task=="lexical" else "speaker_id"; split_col="split"
    if task=="speaker":
        if not speaker_probe_available:return np.nan,np.nan
        base=base[base.pair_id.isin(speaker_pairs)]; base["speaker_probe_split"]=base.pair_id.map(speaker_split_map); split_col="speaker_probe_split"
    else:
        if not lexical_available:return np.nan,np.nan
        base=base.merge(pair_meta[["pair_id","word_label"]],on="pair_id",how="left"); base=base[base.word_label.isin(word_labels)]
    scores=[]
    for source,target in (("IHM","IHM"),("SDM","SDM"),("IHM","SDM"),("SDM","IHM")):
        met,_,_=domain_probe(base,label_col,split_col,source,target,pipeline_for(method))
        scores.append(met["macro_f1"] if met else np.nan)
    return np.nanmean(scores[:2]),np.nanmean(scores[2:])

normalization_rows=[]; normalization_models={}
for method,representation in (("Raw","raw"),("CMN","cmn"),("CMVN","cmvn"),("Robust","raw"),("PCA whiten","raw")):
    mic_met,val_mic,model=eval_mic_method(method,representation); sp_same,sp_cross=eval_task_method(method,representation,"speaker"); lx_same,lx_cross=eval_task_method(method,representation,"lexical")
    normalization_rows.append({"representation":method,"validation_mic_accuracy":val_mic,"mic_accuracy":mic_met["balanced_accuracy"],"mic_f1":mic_met["macro_f1"],
        "speaker_same_channel_f1":sp_same,"speaker_cross_channel_f1":sp_cross,"lexical_same_channel_f1":lx_same,"lexical_cross_channel_f1":lx_cross})
    normalization_models[method]=model

def matrix_power(cov,power,eps=1e-5):
    values,vectors=np.linalg.eigh(cov); return (vectors*np.maximum(values,eps)**power)@vectors.T
def coral_fit(source,target):
    ms,mt=source.mean(0),target.mean(0); cs=np.cov(source,rowvar=False)+1e-3*np.eye(source.shape[1]); ct=np.cov(target,rowvar=False)+1e-3*np.eye(source.shape[1])
    return ms,mt,matrix_power(cs,-.5)@matrix_power(ct,.5)

coral_cross=[]
if speaker_probe_available:
    base=raw[raw.pair_id.isin(speaker_pairs)].copy(); base["speaker_probe_split"]=base.pair_id.map(speaker_split_map)
    for source,target in (("IHM","SDM"),("SDM","IHM")):
        tr=(base.speaker_probe_split=="train")&(base.channel==source); adapt=(base.speaker_probe_split=="train")&(base.channel==target); te=(base.speaker_probe_split=="test")&(base.channel==target)
        scaler=StandardScaler().fit(base.loc[tr,feature_names]); xs=scaler.transform(base.loc[tr,feature_names]); xt=scaler.transform(base.loc[adapt,feature_names]); xe=scaler.transform(base.loc[te,feature_names])
        ms,mt,A=coral_fit(xs,xt); mapped=(xs-ms)@A+mt
        clf=LogisticRegression(max_iter=2500,class_weight="balanced",random_state=SEED).fit(mapped,base.loc[tr,"speaker_id"])
        coral_cross.append(f1_score(base.loc[te,"speaker_id"],clf.predict(xe),average="macro"))
normalization_rows.append({"representation":"CORAL (UDA)","validation_mic_accuracy":np.nan,"mic_accuracy":np.nan,"mic_f1":np.nan,
    "speaker_same_channel_f1":np.nan,"speaker_cross_channel_f1":np.mean(coral_cross) if coral_cross else np.nan,
    "lexical_same_channel_f1":np.nan,"lexical_cross_channel_f1":np.nan})
normalization_results=pd.DataFrame(normalization_rows); normalization_results.to_csv(RESULTS_DIR/"normalization_results.csv",index=False); display(normalization_results)

fig,axes=plt.subplots(1,2,figsize=(13,4.5)); valid=normalization_results.dropna(subset=["mic_accuracy"])
axes[0].bar(valid.representation,valid.mic_accuracy); axes[0].axhline(.5,color="black",ls="--",lw=.8); axes[0].set(ylim=(0,1),ylabel="Balanced accuracy",title="Microphone leakage"); axes[0].tick_params(axis="x",rotation=25)
axes[1].bar(normalization_results.representation,normalization_results.speaker_cross_channel_f1); axes[1].set(ylim=(0,1),ylabel="Macro-F1",title="Cross-channel speaker information"); axes[1].tick_params(axis="x",rotation=25)
plt.tight_layout(); fig.savefig(FIGURE_DIR/"normalization_tradeoff.png",dpi=200); plt.show()
best_norm=valid.sort_values("validation_mic_accuracy",ascending=True).iloc[0]
raw_norm=valid[valid.representation=="Raw"].iloc[0]
method_to_rep={"Raw":"raw","CMN":"cmn","CMVN":"cmvn","Robust":"raw","PCA whiten":"raw"}
for label,method in (("raw", "Raw"),("best_normalized",best_norm.representation)):
    frame=feature_df[feature_df.representation==method_to_rep[method]].copy(); frame["target"]=(frame.channel=="SDM").astype(int); te=frame.split=="test"
    prediction_registry[f"{label}_microphone"]=(frame.loc[te,"target"].to_numpy(),normalization_models[method].predict(frame.loc[te,feature_names]))
print(f"Lowest measured mic leakage: {best_norm.representation} at {best_norm.mic_accuracy:.1%}; raw was {raw_norm.mic_accuracy:.1%}. Downstream values must be considered jointly.")


## 15. Pair-invariant representation learning

The encoder sees engineered features only. Its task head predicts speaker ID in the closed-set diagnostic, while a cosine
penalty pulls matched IHM/SDM embeddings together. Hyperparameter selection uses validation pairs; test performance is
reported only after selection. A separate frozen-embedding logistic probe measures residual microphone information.


In [ ]:
from torch import nn
import torch.nn.functional as F

class PairEncoder(nn.Module):
    def __init__(self,input_dim,n_classes):
        super().__init__()
        self.encoder=nn.Sequential(nn.Linear(input_dim,128),nn.ReLU(),nn.Dropout(.15),nn.Linear(128,64),nn.ReLU(),nn.Linear(64,32))
        self.task_head=nn.Linear(32,n_classes)
    def forward(self,x):
        z=self.encoder(x); return self.task_head(z),z

def pair_loss(a,b): return (1-F.cosine_similarity(a,b)).mean()

invariant_results=pd.DataFrame(); embedding_payloads={}
if speaker_probe_available:
    sp=feature_df[(feature_df.representation=="raw")&feature_df.pair_id.isin(speaker_pairs)].copy(); sp["probe_split"]=sp.pair_id.map(speaker_split_map)
    label_map={s:i for i,s in enumerate(sorted(eligible_speakers))}
    vectors={(r.pair_id,r.channel):r[feature_names].to_numpy(dtype=np.float32) for _,r in sp.iterrows()}
    pair_label={r.pair_id:label_map[r.speaker_id] for r in pair_meta[pair_meta.pair_id.isin(speaker_pairs)].itertuples()}
    ids_by_split={part:[p for p,s in speaker_split_map.items() if s==part and (p,"IHM") in vectors and (p,"SDM") in vectors] for part in ("train","validation","test")}
    fit_vectors=np.vstack([vectors[(p,c)] for p in ids_by_split["train"] for c in ("IHM","SDM")]); inv_scaler=StandardScaler().fit(fit_vectors)
    def batch_arrays(ids):
        return (inv_scaler.transform(np.vstack([vectors[(p,"IHM")] for p in ids])).astype("float32"),
                inv_scaler.transform(np.vstack([vectors[(p,"SDM")] for p in ids])).astype("float32"),
                np.array([pair_label[p] for p in ids],dtype="int64"))
    arrays={part:batch_arrays(ids) for part,ids in ids_by_split.items()}

    def embed_split(model,part):
        hi,di,y=arrays[part]; model.eval()
        with torch.no_grad():
            lh,zh=model(torch.tensor(hi,device=device)); ld,zd=model(torch.tensor(di,device=device))
        return zh.cpu().numpy(),zd.cpu().numpy(),y,lh.argmax(1).cpu().numpy(),ld.argmax(1).cpu().numpy()

    inv_rows=[]
    for lam in tqdm(LAMBDA_VALUES,desc="Pair-invariant sweep"):
        torch.manual_seed(SEED); model=PairEncoder(len(feature_names),len(label_map)).to(device); optimizer=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-5)
        best_state=None; best_val=np.inf; stale=0; rng=np.random.default_rng(SEED)
        for epoch in range(40 if RUN_MODE!="smoke" else 20):
            model.train(); order=rng.permutation(len(ids_by_split["train"])); hi,di,y=arrays["train"]
            for start in range(0,len(order),64):
                idx=order[start:start+64]; xh=torch.tensor(hi[idx],device=device); xd=torch.tensor(di[idx],device=device); target=torch.tensor(y[idx],device=device)
                oh,zh=model(xh); od,zd=model(xd); loss=(F.cross_entropy(oh,target)+F.cross_entropy(od,target))/2+lam*pair_loss(zh,zd)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            vh,vd,vy=arrays["validation"]
            model.eval()
            with torch.no_grad():
                oh,zh=model(torch.tensor(vh,device=device)); od,zd=model(torch.tensor(vd,device=device)); val=float(((F.cross_entropy(oh,torch.tensor(vy,device=device))+F.cross_entropy(od,torch.tensor(vy,device=device)))/2+lam*pair_loss(zh,zd)).cpu())
            if val<best_val-1e-4:
                best_val=val; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
            else:
                stale+=1
                if stale>=6: break
        model.load_state_dict(best_state)
        zh_tr,zd_tr,ytr,pred_h_tr,pred_d_tr=embed_split(model,"train"); zh_v,zd_v,yv,pred_h_v,pred_d_v=embed_split(model,"validation"); zh,zd,y,pred_h,pred_d=embed_split(model,"test")
        Ztr=np.vstack([zh_tr,zd_tr]); Mtr=np.r_[np.zeros(len(zh_tr)),np.ones(len(zd_tr))]
        probe=Pipeline([("scale",StandardScaler()),("clf",LogisticRegression(max_iter=2000,class_weight="balanced",random_state=SEED))]).fit(Ztr,Mtr)
        Zv=np.vstack([zh_v,zd_v]); Mv=np.r_[np.zeros(len(zh_v)),np.ones(len(zd_v))]; Z=np.vstack([zh,zd]); M=np.r_[np.zeros(len(zh)),np.ones(len(zd))]
        val_task=f1_score(np.r_[yv,yv],np.r_[pred_h_v,pred_d_v],average="macro"); val_mic=balanced_accuracy_score(Mv,probe.predict(Zv))
        task=f1_score(np.r_[y,y],np.r_[pred_h,pred_d],average="macro"); mic=balanced_accuracy_score(M,probe.predict(Z))
        matched=np.mean(1-np.sum(zh*zd,axis=1)/(np.linalg.norm(zh,axis=1)*np.linalg.norm(zd,axis=1)+1e-9))
        perm=np.roll(np.arange(len(zd)),1) if len(zd)>1 else np.arange(len(zd)); unmatched=np.mean(1-np.sum(zh*zd[perm],axis=1)/(np.linalg.norm(zh,axis=1)*np.linalg.norm(zd[perm],axis=1)+1e-9))
        selection_score=val_task-.20*abs(val_mic-.5)
        inv_rows.append({"lambda_pair":lam,"validation_task_f1":val_task,"validation_mic_accuracy":val_mic,"selection_score":selection_score,
            "task_f1":task,"mic_probe_accuracy":mic,"matched_pair_distance":matched,"unmatched_distance":unmatched,"epochs":epoch+1})
        embedding_payloads[lam]={"model":model,"Z_train":Ztr,"Z":Z,"mic":M,"speaker":np.r_[y,y],"pair_ids":np.r_[ids_by_split["test"],ids_by_split["test"]],"task_prediction":np.r_[pred_h,pred_d],"task_truth":np.r_[y,y],"mic_prediction":probe.predict(Z)}
    invariant_results=pd.DataFrame(inv_rows); invariant_results.to_csv(RESULTS_DIR/"invariant_results.csv",index=False); display(invariant_results)
    best_lambda=float(invariant_results.sort_values("selection_score",ascending=False).iloc[0].lambda_pair); best_embedding=embedding_payloads[best_lambda]
    prediction_registry["invariant_task"]=(best_embedding["task_truth"],best_embedding["task_prediction"]); prediction_registry["invariant_mic"]=(best_embedding["mic"],best_embedding["mic_prediction"])
    print(f"Validation-selected λ={best_lambda:g}; test task F1={invariant_results.loc[invariant_results.lambda_pair==best_lambda,'task_f1'].iloc[0]:.3f}; mic probe={invariant_results.loc[invariant_results.lambda_pair==best_lambda,'mic_probe_accuracy'].iloc[0]:.3f}.")
else: print("Invariant model skipped because the closed-set speaker task is unavailable.")


## 16. Representation analysis and the central MicCheck tradeoff

PCA is fit on training rows only and then applied to held-out rows. The scatter plot's desirable region is the upper-left:
low microphone predictability with high retained speaker-task performance. Points should be interpreted with their task
definitions and uncertainty, not as a single leaderboard.


In [ ]:
if speaker_probe_available and not invariant_results.empty:
    test_sp=sp[sp.probe_split=="test"].copy(); train_sp=sp[sp.probe_split=="train"]
    raw_scale=StandardScaler().fit(train_sp[feature_names]); pca=PCA(2,random_state=SEED).fit(raw_scale.transform(train_sp[feature_names])); raw_xy=pca.transform(raw_scale.transform(test_sp[feature_names]))
    Z=best_embedding["Z"]
    inv_pca=PCA(2,random_state=SEED).fit(best_embedding["Z_train"]); inv_xy=inv_pca.transform(Z)
    fig,axes=plt.subplots(2,2,figsize=(13,10))
    for ax,xy,labels,title in ((axes[0,0],raw_xy,test_sp.channel.to_numpy(),"Raw by microphone"),(axes[0,1],inv_xy,np.where(best_embedding["mic"]==0,"IHM","SDM"),"Invariant by microphone"),
                               (axes[1,0],raw_xy,test_sp.speaker_id.to_numpy(),"Raw by speaker"),(axes[1,1],inv_xy,best_embedding["speaker"],"Invariant by speaker")):
        for label in np.unique(labels):
            mask=labels==label; ax.scatter(xy[mask,0],xy[mask,1],s=14,alpha=.65,label=str(label))
        ax.set(title=title,xlabel="PC1",ylabel="PC2");
        if len(np.unique(labels))<=12: ax.legend(fontsize=7,ncol=2)
    plt.tight_layout(); fig.savefig(FIGURE_DIR/"pca_before_after.png",dpi=200); plt.show()

trade=[]
for row in normalization_results.itertuples():
    if np.isfinite(row.mic_accuracy) and np.isfinite(row.speaker_cross_channel_f1): trade.append((row.representation,row.mic_accuracy,row.speaker_cross_channel_f1))
for row in invariant_results.itertuples(): trade.append((f"PairInv λ={row.lambda_pair:g}",row.mic_probe_accuracy,row.task_f1))
if trade:
    fig,ax=plt.subplots(figsize=(9,6))
    for label,x,y in trade:
        ax.scatter(x,y,s=55); ax.annotate(label,(x,y),xytext=(5,5),textcoords="offset points",fontsize=8)
    ax.axvline(.5,color="black",ls="--",lw=.8); ax.set(xlabel="Microphone probe balanced accuracy ↓",ylabel="Speaker-information macro-F1 ↑",title="MicCheck channel-leakage / speech-information tradeoff",xlim=(.45,1.02),ylim=(0,1.02))
    ax.text(.46,.97,"desirable",ha="left",va="top",style="italic"); plt.tight_layout(); fig.savefig(FIGURE_DIR/"central_tradeoff.png",dpi=240); plt.show()

if not invariant_results.empty:
    fig,axes=plt.subplots(1,2,figsize=(12,4.2)); axes[0].plot(invariant_results.lambda_pair,invariant_results.task_f1,"o-",label="task F1"); axes[0].plot(invariant_results.lambda_pair,invariant_results.mic_probe_accuracy,"o-",label="mic probe")
    axes[0].set(xlabel="Pair-loss weight λ",ylabel="Score",title="Invariant sweep",ylim=(0,1)); axes[0].legend()
    axes[1].plot(invariant_results.lambda_pair,invariant_results.matched_pair_distance,"o-",label="matched"); axes[1].plot(invariant_results.lambda_pair,invariant_results.unmatched_distance,"o-",label="unmatched")
    axes[1].set(xlabel="Pair-loss weight λ",ylabel="Cosine distance",title="Embedding distances"); axes[1].legend(); plt.tight_layout(); fig.savefig(FIGURE_DIR/"invariant_sweep.png",dpi=200); plt.show()


## 17. Exploratory counterfactual channel transformation

An aggregate log-amplitude response is estimated from **training-speaker pairs only** using Welch PSDs, median aggregation,
and frequency smoothing. Applying the corresponding regularized FIR filter to held-out IHM speech creates a pseudo-SDM
diagnostic. This changes spectral coloration but cannot recreate room geometry, reverberation, source position, or noise;
prediction shifts therefore suggest sensitivity to channel characteristics and are not standalone causal proof.


In [ ]:
cf_train=pair_meta[pair_meta.split=="train"].sample(min(COUNTERFACTUAL_TRAIN_PAIRS,(pair_meta.split=="train").sum()),random_state=SEED)
cf_test=pair_meta[pair_meta.split=="test"].sample(min(3,(pair_meta.split=="test").sum()),random_state=SEED)
cf_psd={"IHM":{},"SDM":{}}; cf_audio={}
for config,column,channel in (("ihm","ihm_audio_id","IHM"),("sdm","sdm_audio_id","SDM")):
    combined=pd.concat([cf_train,cf_test]).drop_duplicates("pair_id"); wanted=dict(zip(combined[column],combined.pair_id)); test_ids=set(cf_test.pair_id)
    for audio_id,y,sr in stream_selected(config,wanted):
        pair_id=wanted[audio_id]; f,psd=signal.welch(y,sr,nperseg=1024); cf_psd[channel][pair_id]=psd
        if pair_id in test_ids: cf_audio[(pair_id,channel)]=y.copy()
        del y
training_differences=[]
for pair_id in cf_train.pair_id:
    if pair_id in cf_psd["IHM"] and pair_id in cf_psd["SDM"]:
        training_differences.append(.5*(np.log(cf_psd["SDM"][pair_id]+1e-10)-np.log(cf_psd["IHM"][pair_id]+1e-10)))
if not training_differences: raise RuntimeError("No training audio pairs available for counterfactual response")
log_gain=ndimage.gaussian_filter1d(np.median(np.vstack(training_differences),axis=0),sigma=3)
gain=np.exp(np.clip(log_gain,-3,3)); taps=signal.firwin2(513,f/(16000/2),gain)

def pseudo_sdm(y):
    out=signal.fftconvolve(y,taps,mode="same").astype("float32"); peak=np.max(np.abs(out),initial=0)
    return out*(.99/peak) if peak>.99 else out
def vector_for(y): return acoustic_variants(y,16000)["raw"]

cf_rows=[]; raw_mic_model=normalization_models["Raw"]
for row in cf_test.itertuples():
    if (row.pair_id,"IHM") not in cf_audio or (row.pair_id,"SDM") not in cf_audio: continue
    hi=cf_audio[(row.pair_id,"IHM")]; sd=cf_audio[(row.pair_id,"SDM")]; pseudo=pseudo_sdm(hi)
    vectors=np.vstack([vector_for(hi),vector_for(pseudo),vector_for(sd)])
    probabilities=raw_mic_model.predict_proba(pd.DataFrame(vectors,columns=feature_names))[:,1]
    record={"pair_id":row.pair_id,"speaker_id":row.speaker_id,"text":row.text,"p_sdm_ihm":probabilities[0],"p_sdm_pseudo":probabilities[1],"p_sdm_real":probabilities[2],"lexical_prediction_changed":np.nan}
    if lexical_available and row.word_label in word_labels and "IHM→SDM" in lexical_models:
        task_model=lexical_models["IHM→SDM"]; predictions=task_model.predict(pd.DataFrame(vectors,columns=feature_names)); record["lexical_prediction_changed"]=bool(predictions[1]!=predictions[0]); record["lexical_predictions"]=" / ".join(map(str,predictions))
    cf_rows.append(record)
    print(f"{row.meeting_id} | {row.speaker_id} | {row.text}\nOriginal IHM / pseudo-SDM / actual SDM")
    display(Audio(hi,rate=16000)); display(Audio(pseudo,rate=16000)); display(Audio(sd,rate=16000))
    fig,axes=plt.subplots(1,3,figsize=(15,3.8))
    for ax,y,label in zip(axes,(hi,pseudo,sd),("Original IHM","Pseudo-SDM","Actual SDM")):
        L=logmel(y); ax.imshow(L,origin="lower",aspect="auto",extent=[0,len(y)/16000,0,8000],cmap="magma"); ax.set(title=label,xlabel="Time (s)",ylabel="Frequency (Hz)")
    plt.tight_layout(); fig.savefig(FIGURE_DIR/f"counterfactual_{row.pair_id}.png",dpi=200); plt.show()
counterfactual_results=pd.DataFrame(cf_rows); counterfactual_results.to_csv(RESULTS_DIR/"counterfactual_results.csv",index=False); display(counterfactual_results)
if not counterfactual_results.empty:
    means=counterfactual_results[["p_sdm_ihm","p_sdm_pseudo","p_sdm_real"]].mean(); fig,ax=plt.subplots(figsize=(7,4.5)); ax.bar(["IHM","Pseudo-SDM","Real SDM"],means)
    ax.set(ylim=(0,1),ylabel="Mean P(SDM)",title="Held-out counterfactual microphone probability"); plt.tight_layout(); fig.savefig(FIGURE_DIR/"counterfactual_probability.png",dpi=200); plt.show()
fig,ax=plt.subplots(figsize=(9,4)); ax.plot(f,20*np.log10(gain+1e-10)); ax.axhline(0,color="black",lw=.8); ax.set(xlabel="Frequency (Hz)",ylabel="Estimated amplitude gain (dB)",title="Training-only aggregate IHM→SDM spectral response")
plt.tight_layout(); fig.savefig(FIGURE_DIR/"estimated_channel_response.png",dpi=200); plt.show()


## 18. Bootstrap uncertainty

Percentile intervals resample held-out predictions. They quantify item-level uncertainty, not uncertainty over the AMI
population of speakers. With few held-out speakers they can be optimistic, so speaker counts remain essential context.


In [ ]:
def bootstrap_ci(y_true,y_pred,metric,n=N_BOOTSTRAPS,seed=SEED):
    y_true=np.asarray(y_true); y_pred=np.asarray(y_pred); rng=np.random.default_rng(seed); values=[]
    for _ in range(n):
        idx=rng.integers(0,len(y_true),len(y_true))
        try: values.append(metric(y_true[idx],y_pred[idx]))
        except Exception: pass
    return (np.quantile(values,.025),np.quantile(values,.975)) if values else (np.nan,np.nan)

ci_rows=[]
for name,(truth,pred) in prediction_registry.items():
    metric=(lambda a,b:balanced_accuracy_score(a,b)) if name.startswith("mic_") or name=="invariant_mic" or "microphone" in name else (lambda a,b:f1_score(a,b,average="macro"))
    lo,hi=bootstrap_ci(truth,pred,metric); ci_rows.append({"metric":name,"point":metric(truth,pred),"ci_95_low":lo,"ci_95_high":hi,"bootstrap_iterations":N_BOOTSTRAPS})
uncertainty=pd.DataFrame(ci_rows); uncertainty.to_csv(RESULTS_DIR/"bootstrap_intervals.csv",index=False); display(uncertainty)


## 19. Final comparison dashboard

Missing values mean the corresponding experiment could not be run or was not scientifically defined; they are never
filled with guesses. Pair-invariant task F1 is listed under the closed-set speaker-information column, while its
cross-channel cell stays empty because its task head was jointly trained on both conditions.


In [ ]:
final_summary=normalization_results.copy(); final_summary["paired_embedding_distance"]=np.nan
if not invariant_results.empty:
    invariant_dashboard=[]
    for row in invariant_results.itertuples():
        invariant_dashboard.append({"representation":f"PairInv λ={row.lambda_pair:g}","mic_accuracy":row.mic_probe_accuracy,"mic_f1":np.nan,
            "speaker_same_channel_f1":row.task_f1,"speaker_cross_channel_f1":np.nan,"lexical_same_channel_f1":np.nan,
            "lexical_cross_channel_f1":np.nan,"paired_embedding_distance":row.matched_pair_distance})
    final_summary=pd.concat([final_summary,pd.DataFrame(invariant_dashboard)],ignore_index=True)
final_summary=final_summary[["representation","mic_accuracy","mic_f1","speaker_same_channel_f1","speaker_cross_channel_f1","lexical_same_channel_f1","lexical_cross_channel_f1","paired_embedding_distance"]]
final_summary.to_csv(RESULTS_DIR/"final_summary.csv",index=False); display(final_summary)

baseline_export=mic_results.copy()
for task,table in (("speaker_closed_set",speaker_results),("lexical",lexical_results)):
    if not table.empty:
        add=table.copy(); add.insert(0,"experiment",task); add.insert(1,"model","Logistic Regression"); add.insert(2,"representation","Raw")
        baseline_export=pd.concat([baseline_export,add],ignore_index=True,sort=False)
baseline_export.to_csv(RESULTS_DIR/"baseline_results.csv",index=False)


## 20. Automatically generated findings and hypothesis checks

The following narrative is created only from computed values. Thresholds are declared in code so “supported” is not chosen
after reading the answer: strong microphone evidence ≥70%, moderate ≥58%; material degradation or leakage reduction is five
percentage points; task preservation permits at most a five-point loss.


In [ ]:
raw_row=normalization_results[normalization_results.representation=="Raw"].iloc[0]
lines=["### Computed findings",f"1. **Channel leakage.** Raw acoustic features yielded {raw_row.mic_accuracy:.1%} speaker-independent microphone balanced accuracy."]
if np.isfinite(raw_row.speaker_same_channel_f1) and np.isfinite(raw_row.speaker_cross_channel_f1):
    drop=raw_row.speaker_same_channel_f1-raw_row.speaker_cross_channel_f1
    lines.append(f"2. **Cross-channel speaker probe.** Mean macro-F1 changed from {raw_row.speaker_same_channel_f1:.3f} same-channel to {raw_row.speaker_cross_channel_f1:.3f} cross-channel (degradation {drop:+.3f}).")
else: drop=np.nan; lines.append("2. **Cross-channel speaker probe.** Insufficient data for a stable closed-set probe.")
candidate=best_norm
task_change=(candidate.speaker_cross_channel_f1-raw_row.speaker_cross_channel_f1) if np.isfinite(candidate.speaker_cross_channel_f1) and np.isfinite(raw_row.speaker_cross_channel_f1) else np.nan
lines.append(f"3. **Normalization.** {candidate.representation} had the lowest measured microphone accuracy ({candidate.mic_accuracy:.1%}), a change of {candidate.mic_accuracy-raw_row.mic_accuracy:+.1%} from raw; cross-channel speaker-F1 change was {task_change:+.3f}." if np.isfinite(task_change) else f"3. **Normalization.** {candidate.representation} had the lowest measured microphone accuracy ({candidate.mic_accuracy:.1%}), a change of {candidate.mic_accuracy-raw_row.mic_accuracy:+.1%} from raw.")
if not invariant_results.empty:
    chosen=invariant_results.loc[invariant_results.lambda_pair==best_lambda].iloc[0]
    lines.append(f"4. **Pair invariance.** Validation selected λ={best_lambda:g}; test task F1={chosen.task_f1:.3f}, microphone-probe accuracy={chosen.mic_probe_accuracy:.1%}, and matched cosine distance={chosen.matched_pair_distance:.3f}.")
if not counterfactual_results.empty:
    means=counterfactual_results[["p_sdm_ihm","p_sdm_pseudo","p_sdm_real"]].mean()
    lines.append(f"5. **Spectral diagnostic.** Mean P(SDM) moved from {means.p_sdm_ihm:.1%} on original IHM to {means.p_sdm_pseudo:.1%} on pseudo-SDM; real SDM was {means.p_sdm_real:.1%}. This suggests classifier sensitivity to the estimated coloration, without isolating all causal channel factors.")

h1="supported" if raw_row.mic_accuracy>=.70 else "partially supported" if raw_row.mic_accuracy>=.58 else "not supported"
h2="not evaluated" if not np.isfinite(drop) else "supported" if drop>=.05 else "partially supported" if drop>0 else "not supported"
leak_reduction=raw_row.mic_accuracy-candidate.mic_accuracy
h3="supported" if leak_reduction>=.05 and (not np.isfinite(task_change) or task_change>=-.05) else "partially supported" if leak_reduction>0 else "not supported"
if invariant_results.empty: h4="not evaluated"
else:
    base_inv=invariant_results.iloc[(invariant_results.lambda_pair-0).abs().argmin()]; chosen=invariant_results.loc[invariant_results.lambda_pair==best_lambda].iloc[0]
    h4="supported" if chosen.matched_pair_distance<base_inv.matched_pair_distance and chosen.task_f1>=base_inv.task_f1-.05 else "partially supported" if chosen.matched_pair_distance<base_inv.matched_pair_distance else "not supported"
lines += ["","### Pre-declared hypothesis assessment",f"- **H1:** {h1}",f"- **H2:** {h2}",f"- **H3:** {h3}",f"- **H4:** {h4}"]
display(Markdown("\n\n".join(lines)))


## 21. Limitations

- IHM and SDM differ by more than microphone hardware: distance, room acoustics, reverberation, noise, placement, and
  potentially gain control all change.
- AMI is meeting speech and does not represent every language, speaking style, room, device, or population.
- This bounded streamed subset may have limited statistical power and speaker/meeting imbalance.
- Engineered acoustic features are interpretable and lightweight, but are not modern foundation-model representations.
- Closed-set speaker classification is a proxy for retained speaker information, not an unseen-speaker recognition claim.
- Single-word lexical classification covers only frequent short prompts present across disjoint speakers.
- Item-bootstrap intervals do not replace uncertainty over independently sampled speakers or meetings.
- The estimated spectral response is linear and aggregate; an SDM recording is emphatically not merely a filtered IHM
  recording.
- CORAL uses unlabeled target-domain statistics from training pairs and is not directly comparable to strict source-only
  methods without that qualification.

MicCheck should therefore be read as a controlled probe of recording-condition sensitivity and shortcut information—not
as proof of causal microphone bias.

## 22. Future research

Useful next steps are grouped speaker-level cross-validation, a second paired-channel corpus, speaker-verification metrics
on entirely unseen identities, nonlinear but interpretable room-response models, adversarial channel probes, and external
microphone transfer. A preregistered analysis with more independent rooms and speakers would make the tradeoff estimates
substantially stronger.


## 23. Export results

The results directory contains the paired manifest, audit, exclusions, feature parquet, experiment tables, confidence
intervals, counterfactual diagnostics, and publication-resolution figures. The archive is created without automatically
triggering a browser download.


In [ ]:
archive_base=PROJECT_DIR / "MicCheck_results"
archive_path=Path(shutil.make_archive(str(archive_base),"zip",root_dir=RESULTS_DIR))
print("Saved results:")
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file(): print(" -",path.relative_to(RESULTS_DIR))
print(f"\nArchive: {archive_path} ({archive_path.stat().st_size/2**20:.1f} MiB)")


The generated `MicCheck_results.zip` archive is saved in the repository root for optional sharing or archival.
